# Iberdrola Datathon: EV Infrastructure Roadmap

**Team:** Colomany

## General Description
This notebook provides a detailed end-to-end pipeline for the **Iberdrola Datathon**. We transform raw geospatial road data into a discrete point-based network enriched with traffic intensity, electrical capacity, and proximity to existing infrastructure. Finally, we solve a grid-aware optimization model to identify the best locations for new ultra-fast EV chargers.

## Table of Contents
1. [Phase 0: Environment Setup & Data Acquisition](#phase-0)
2. [Phase 1: Demand Forecasting (EV Registrations)](#phase-1)
3. [Phase 2: Spatial Backbone Foundation](#phase-2)
4. [Phase 3: Grid-Aware Optimization](#phase-3)
5. [Phase 4: Results & Visualization](#phase-4)

---


# Phase 0: Environment Setup & Data Acquisition <a name="phase-0"></a>


### 0.1 Repository Setup
**Procedure**: Cloning the project repository to the local environment. Important for Google Colab sessions.

In [1]:
!git clone https://github.com/JJR9903/Iberdrola-Datathon.git

Cloning into 'Iberdrola-Datathon'...
remote: Enumerating objects: 272, done.
remote: Counting objects: 100% (272/272), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 272 (delta 145), reused 207 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (272/272), 1.39 MiB | 7.21 MiB/s, done.
Resolving deltas: 100% (145/145), done.


### 0.2 Working Directory
**Procedure**: Changing the current working directory to the project root. Important for Google Colab sessions.


In [3]:
cd "/content/Iberdrola-Datathon"

/content/Iberdrola-Datathon


### 0.3 Library & Script Imports
**Procedure**: Importing essential libraries (Pandas, GeoPandas, etc.) and custom processing functions.
**Details**:
- `discretize_backbone_roads`: To convert LineStrings to Points.
- `map_traffic_to_points`: To project traffic data.
- `assign_nearest_charging_stations`/`gas_stations`: Proximity analysis.
**Main Assumptions**: All dependencies listed in `pyproject.toml` are installed. (For Google Colab sessions, these dependencies are usually installed by default)


In [3]:
!pip install pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.6 MB/s eta 0:00:00


In [ ]:
cd "/content/Iberdrola-Datathon"

In [1]:
import geopandas as gpd
import pandas as pd
import folium
import os
import numpy as np
import tomllib
import sys
import polars as pl
import statsmodels.api as sm
import pmdarima as pm
from datetime import date
from dateutil.relativedelta import relativedelta



# Change directory to root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Add scripts directory to path to allow module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from scripts.sync_cloud import sync_standardized_data
from scripts.processing.create_backbone_foundation import (
    discretize_backbone_roads,
    map_traffic_to_points,
    assign_nearest_charging_stations,
    assign_nearest_gas_stations,
    assign_grid_capacity
)


### 0.4 Cloud Data Synchronization

> **Note**: A previous data extraction and standardization process was performed using the project's specialized scripts (scripts/01_acquisition.py for downloading the raw data files, bronze layer) and (scripts/02_standardization.py to standardize each dataset to some specific columns and save them in a parquet format that saves space in memory, Silver layers). For reproducibility and efficiency, this notebook starts by synchronizing these processed **Silver Data** files directly from the cloud to ensure a consistent baseline for analysis.

**Procedure**: Using `sync_standardized_data()` to ensure all required Parquet files are present in the local `data/` directory.
**Datasets Used**: Downloads `roads.parquet`, `traffic.parquet`, `chargers.parquet`, `electric_capacity.parquet`, `vehicle_registrations.parquet`, `gas_stations.parquet`.
**Main Assumptions**:
- *Assumption*: Cloud storage is accessible.
- *Rationale*: This ensures the "Silver Layer" (standardized data) is available for analysis.


In [2]:
# 1. Sync data (Ensures silver layer is present)
sync_standardized_data()


=== Iberdrola Datathon: Cloud Data Sync ===

 - Downloading roads.parquet from cloud...
 - Downloading traffic.parquet from cloud...
 - Downloading chargers.parquet from cloud...
 - Downloading electric_capacity.parquet from cloud...
 - Downloading vehicle_registrations.parquet from cloud...
 - Downloading gas_stations.parquet from cloud...

✅ Sync complete: 6 succeeded, 0 failed.


True

### 0.5 Configuration Loading
**Procedure**: Loading parameters from `config.toml` for the backbone foundation step.
**Datasets Used**: `config.toml`.
**Main Assumptions**: The config file contains a `backbone_foundation` section with valid paths and parameters.


In [3]:
# 2. Load Config
with open('config.toml', 'rb') as f:
    config = tomllib.load(f)
params = config['steps']['backbone_foundation']


### 0.6 Data Loading
**Procedure**: Reading the standardized Parquet files into GeoDataFrames and Polars DataFrames.
**Datasets Used**:
- `gdf_roads`: Road backbone segments.
- `gdf_traffic`: Traffic intensity data.
- `gdf_chargers`: Existing EV chargers.
- `gdf_gas`: Traditional gas stations.
- `gdf_capacity`: Electric grid substation capacity.
- `pl_registrations`: Historical vehicle registrations.
**Main Assumptions**: File paths in `config.toml` are correct.


In [4]:
# 3. Load Standardized Data

print("\n🚀 Loading Standardized Data...")
gdf_roads = gpd.read_parquet(params['roads_path'])
gdf_traffic = gpd.read_parquet(params['traffic_path'])
gdf_chargers = gpd.read_parquet(params['chargers_path'])
gdf_gas = gpd.read_parquet(params['gas_stations_path'])
gdf_capacity = gpd.read_parquet(params['capacity_path'])
pl_registrations = pl.read_parquet('data/standardized/vehicle_registrations.parquet')

print(f" - Roads: {len(gdf_roads)} segments")
print(f" - Traffic: {len(gdf_traffic)} segments")
print(f" - Chargers: {len(gdf_chargers)} sites")
print(f" - Gas Stations: {len(gdf_gas)} sites")
print(f" - Electric Capacity: {len(gdf_capacity)} sites")
print(f" - Vehicle Registrations: {len(pl_registrations)}")

print("\n✅ Data loaded correctly.")


🚀 Loading Standardized Data...
 - Roads: 1591 segments
 - Traffic: 337976 segments
 - Chargers: 1298 sites
 - Gas Stations: 12204 sites
 - Electric Capacity: 2193 sites
 - Vehicle Registrations: 12969460

✅ Data loaded correctly.


**Procedure**: Verifying the current working directory.


In [5]:
os.getcwd()

'/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon'

**Procedure**: Visualizing the first few rows of the roads dataset.


In [6]:
gdf_roads.head()

,road_id,road_name,road_type,length_m,geometry
0,ID_190000,A-7S,Autopista peaje,5322.812053,"LINESTRING Z (332545.402 4043245.216 60.099, 3..."
1,ID_190001,A-7S,Autopista libre\Autovía,450.830611,"LINESTRING Z (327350.883 4042557.782 51.345, 3..."
2,ID_190002,A-7S,Autopista libre\Autovía,22389.956273,"LINESTRING Z (326926.286 4042684.128 57.736, 3..."
3,ID_190003,A-7S,Autopista peaje,2587.326470,"LINESTRING Z (308321.898 4034405.237 35.265, 3..."
4,ID_190004,A-7S,Multicarril,20452.485224,"LINESTRING Z (305774.262 4034160.476 42.619, 3..."


**Procedure**: Visualizing the traffic dataset segments.


In [7]:
gdf_traffic.head()

,traffic_segment_id,geometry,total_20240331,short_20240331,medio_20240331,total_20240824,short_20240824,medio_20240824,total_20240827,short_20240827,medio_20240827,total_20241016,short_20241016,medio_20241016,total_20241019,short_20241019,medio_20241019,total_max,short_max,medio_max
0,431620080527,"LINESTRING (828792.863 4546293.054, 828793.558...",11979.36,1157.48,6033.28,16396.96,1881.43,8791.87,12653.77,1455.42,7159.17,8617.87,988.21,5578.68,11012.58,1010.20,7540.35,16396.96,1881.43,8791.87
1,450380176750,"LINESTRING (421324.886 4446687.458, 421311.987...",1814.23,1267.17,512.25,2068.41,1546.80,440.46,2286.39,1707.01,563.55,2862.48,1961.31,891.49,2412.32,1773.52,622.02,2862.48,1961.31,891.49
2,450410188347,"LINESTRING (408328.106 4452615.976, 408335.5 4...",2206.83,1848.81,321.93,2057.66,1809.55,214.73,3249.74,2717.22,508.92,4237.48,3402.05,819.92,3790.37,3204.91,559.32,4237.48,3402.05,819.92
3,451210187556,"LINESTRING (462769.447 4420963.895, 462775.306...",12426.95,665.20,6615.70,3345.17,801.43,1659.74,3973.91,784.30,2606.89,3188.89,853.67,2028.52,3903.08,842.49,2766.34,12426.95,853.67,6615.70
4,462110000438,"LINESTRING (743184.14 4314154.43, 743242.28 43...",21933.22,172.69,17106.84,44760.92,242.87,33344.83,26237.97,269.57,20655.05,14154.96,222.97,11740.40,23880.43,221.92,20112.69,44760.92,269.57,33344.83


**Procedure**: Visualizing existing charging stations.


In [8]:
gdf_chargers.head()

,site_id,site_name,city,province,max_power_kw,geometry,charger_count
0,0ABYOR885E4US6RFRJFV,TotalEnergies - Hotel Mirasierra,None,None,150.0,POINT (451056.608 4559802.443),4
1,0MSSBLUV4UTE7AE2J1B6,"Repsol ES, CRED Calahorra M.I.",None,None,120.0,POINT (584948.394 4682782.607),2
2,0OW3YBKDKJUTPB7ALJGO,L-VielhaEMijaran-005,None,None,110.0,POINT (811005.244 4735727.009),2
3,0S9SNBBYAX5TI0YO5JKW,"Repsol ES, La Junquera",None,None,180.0,POINT (984035.631 4710815.85),2
4,0U2UXOGZDVMXLO61BBCZ,"Repsol ES, CRED Lopidana Dcho",None,None,150.0,POINT (523465.844 4747233.793),2


**Procedure**: Visualizing traditional gas stations.


In [9]:
gdf_gas.head()

,station_id,city,province,geometry
0,4375,Abengibre,ALBACETE,POINT (626121.558 4341254.741)
1,5122,Alatoz,ALBACETE,POINT (643017.084 4329218.999)
2,12054,Albacete,ALBACETE,POINT (601062.716 4323495.143)
3,10765,Albacete,ALBACETE,POINT (597999.676 4315794.866)
4,4438,Albacete,ALBACETE,POINT (599900.247 4317156.738)


**Procedure**: Visualizing raw registration data.


In [10]:
pl_registrations.head()

date,brand,propulsion
date,str,str
2015-01-02,"""VOLKSWAGEN""","""Diesel"""
2015-01-02,"""VOLKSWAGEN""","""Diesel"""
2015-01-02,"""LEXUS""","""Gasolina"""
2015-01-02,"""BMW""","""Diesel"""
2015-01-02,"""AUDI""","""Diesel"""


**Procedure**: Visualizing electric grid capacity (substations).


In [11]:
gdf_capacity.head()

,substation,grid_operator,capacity_kw,company,province,city,row_id,geometry
0,58038225,R1-299,0.0,Endesa,4,4079,R1-299_58038225,POINT (535771.67 4074262.07)
1,58037946,R1-299,0.0,Endesa,4,4903,R1-299_58037946,POINT (528430.08 4072234.8)
2,533105218,R1-299,0.0,Endesa,11,11008,R1-299_533105218,POINT (279539.31 4010543.81)
3,531021233,R1-299,3090.0,Endesa,11,11014,R1-299_531021233,POINT (224745.22 4020492.76)
4,532628547,R1-299,0.0,Endesa,11,11020,R1-299_532628547,POINT (220439.66 4061203.29)


# Phase 1: Demand Forecasting (EV Registrations) <a name="phase-1"></a>

In this phase, we analyze historical vehicle registration data from DGT to forecast the penetration of Electric Vehicles (EV) in Spain by 2027.

**Methodology Reference**:
- Inspired by the [Laboratorio de Datos GitHub Repo](https://github.com/Admindatosgobes/Laboratorio-de-Datos/tree/main/Data%20Science/Ruta%20a%20la%20electrificaci%C3%B3n%20de%20la%20Movilidad)
- and the [Ruta a la electrificación Colab Notebook](https://colab.research.google.com/github/Admindatosgobes/Laboratorio-de-Datos/blob/main/Data%20Science/Ruta%20a%20la%20electrificaci%C3%B3n%20de%20la%20Movilidad/Codigo/Notebook.ipynb#scrollTo=0b9269b0-d9a3-46d1-8265-ed8a17f086ff)


### 1.1 Data Filtering
**Procedure**: Filtering the registrations to focus on the 2015-2025 period.
**Datasets Used**: `pl_registrations`.
**Main Assumptions**:
- *Assumption*: 2015 is a sufficient starting point for EV trends.
- *Rationale*: Older data may not reflect modern adoption rates.


In [12]:
pl_registrations = pl_registrations.filter(
   ( pl.col('date') >= date(2015, 1, 1) ) & (pl.col('date') < date(2026, 1, 1))
)

In [13]:
non_ev_registrations = pl_registrations.filter(pl.col('propulsion') != 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')
).sort("date").to_pandas().set_index('date')

ev_registrations = pl_registrations.filter(pl.col('propulsion') == 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')
).sort("date").to_pandas().set_index('date')


### 1.2 EV Demand Forecast
**Procedure**: Using **Auto-ARIMA** to find optimal parameters and fitting a **SARIMAX** model to the **log of EV registrations**.
**Datasets Used**: `ev_registrations`.
**Main Assumptions**:
- *Assumption*: Log-transformation handles exponential growth better.
- *Rationale*: Registration trends often show multiplicative seasonality.
- *Assumption*: 12-month seasonality (m=12).


In [14]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_EV} & seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(ev_registrations['registrations']),
    order=best_order_EV,
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))
EV_fc_mean = round(np.exp(EV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': EV_fc_mean
}, index=forecast_dates)


ev_registrations = pd.concat([ev_registrations, df_ev_forecast]).rename(columns={'registrations':'ev_registrations'})

ev_registrations.tail(12)

using a SARIMAX with order=(1, 0, 2) & seasonal_order=(1, 0, 1, 12)


/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rin

,ev_registrations
2027-01-01,10993.0
2027-02-01,12314.0
2027-03-01,15170.0
2027-04-01,11167.0
2027-05-01,13312.0
2027-06-01,16299.0
2027-07-01,14224.0
2027-08-01,11609.0
2027-09-01,16695.0
2027-10-01,16779.0


### 1.3 Non EV Demand Forecast
**Procedure**: Using **Auto-ARIMA** to find optimal parameters and fitting a **SARIMAX** model to the **log of Non EV registrations**.
**Datasets Used**: `non_ev_registrations`.
**Main Assumptions**:
- *Assumption*: Log-transformation handles exponential growth better.
- *Rationale*: Registration trends often show multiplicative seasonality.
- *Assumption*: 12-month seasonality (m=12).


In [15]:
# Selección automática del modelo con Auto-ARIMA
auto_nEV_model = pm.auto_arima(np.log(non_ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_nEV = auto_nEV_model.order
best_seasonal_order_nEV = auto_nEV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_nEV} & seasonal_order={best_seasonal_order_nEV} with drift")

nEV_model = sm.tsa.statespace.SARIMAX(
    np.log(non_ev_registrations['registrations']),
    trend='c' if auto_nEV_model.with_intercept==True else 'n',
    order=best_order_nEV,
    seasonal_order=best_seasonal_order_nEV
)
nEV_results = nEV_model.fit(disp=False)

forecast_steps = 24
nEV_model_forecast = nEV_results.get_forecast(steps=forecast_steps)

nEV_model_forecast = (nEV_model_forecast.summary_frame(alpha=0.5))
nEV_fc_mean = round(np.exp(nEV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = non_ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': nEV_fc_mean
}, index=forecast_dates)


non_ev_registrations = pd.concat([non_ev_registrations, df_ev_forecast]).rename(columns={'registrations':'non_ev_registrations'})

non_ev_registrations

using a SARIMAX with order=(0, 0, 2) & seasonal_order=(0, 0, 2, 12) with drift


/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:1009: UserWarning: Non-invertible starting seasonal moving average Using zeros as starting parameters.
  warn('Non-invertible starting seasonal moving average'


,non_ev_registrations
2015-01-01,71240.0
2015-02-01,90257.0
2015-03-01,115720.0
2015-04-01,86170.0
2015-05-01,97666.0
...,...
2027-08-01,87452.0
2027-09-01,90111.0
2027-10-01,91402.0
2027-11-01,90980.0


### 1.2 Data Aggregation
**Procedure**: Grouping and counting registrations by month for both EV and Non-EV categories.


In [16]:
vehicle_registrations = ev_registrations.merge(non_ev_registrations,how='inner',left_index=True, right_index=True)
vehicle_registrations

,ev_registrations,non_ev_registrations
2015-01-01,55.0,71240.0
2015-02-01,51.0,90257.0
2015-03-01,162.0,115720.0
2015-04-01,111.0,86170.0
2015-05-01,118.0,97666.0
...,...,...
2027-08-01,11609.0,87452.0
2027-09-01,16695.0,90111.0
2027-10-01,16779.0,91402.0
2027-11-01,17671.0,90980.0


**Procedure**: Calculating total registration counts and current EV percentage.


In [17]:
vehicle_registrations_total = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations["non_ev_registrations"].sum()],
})

vehicle_registrations_total["pct_ev_registrations"] = (
    vehicle_registrations_total["total_ev_registrations"] /
    (vehicle_registrations_total["total_ev_registrations"] + vehicle_registrations_total["total_non_ev_registrations"])
)
vehicle_registrations_total

,total_ev_registrations,total_non_ev_registrations,pct_ev_registrations
0,664995.0,14626597.0,0.043488


**Procedure**: Isolating the 2027 forecast period to determine future EV penetration percentage.


In [18]:
vehicle_registrations_2027 = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["non_ev_registrations"].sum()],
})

vehicle_registrations_2027["pct_ev_registrations"] = (
    vehicle_registrations_2027["total_ev_registrations"] /
    (vehicle_registrations_2027["total_ev_registrations"] + vehicle_registrations_2027["total_non_ev_registrations"])
)
vehicle_registrations_2027

,total_ev_registrations,total_non_ev_registrations,pct_ev_registrations
0,176583.0,1093526.0,0.13903


# Phase 2: Spatial Backbone Foundation <a name="phase-2"></a>

We discretize the road network into equidistant points and enrich them with traffic and infrastructure proximity data.


### 2.1 Road Discretization
**Procedure**: Converting LineString road geometries into discrete points every 200 meters using `discretize_backbone_roads`.
**Datasets Used**: `gdf_roads`.
**Main Assumptions**:
- *Assumption*: `sampling_interval_m` = 200.
- *Rationale*: Provides enough resolution for charging station placement without excessive compute.


In [19]:
params['sampling_interval_m']

200

In [20]:
gdf_points = discretize_backbone_roads(
    gdf_roads,
    sampling_interval_m=params['sampling_interval_m']
)
print(f"Generated {len(gdf_points)} points along the backbone.")
gdf_points.head()

 - Discretizing backbones into points (Interval=200m)...
Generated 147288 points along the backbone.


,backbone_id,road_name,road_type,length_m,geometry,m_ref,point_idx,point_id
0,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (332545.402 4043245.216 60.099),0.0,0,ID_190000_0
1,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (332347.326 4043217.681 69.7),200.0,1,ID_190000_1
2,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (332147.851 4043216.941 68.397),400.0,2,ID_190000_2
3,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (331953.2 4043260.901 71.507),600.0,3,ID_190000_3
4,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (331763.559 4043324.239 74.492),800.0,4,ID_190000_4


### Visualization: Base Point Network
Below we see the result of the discretization. Each red dot represents a sampling point where we will later attach traffic and infrastructure data.

**Visualization**: Displaying the discretized backbone points across Spain.


In [ ]:
m0 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
gdf_plot = gdf_points.to_crs(4326)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='red',
        fill=True
    ).add_to(m0)
m0

### 2.2 Traffic Mapping
**Procedure**: Mapping the closest traffic segments to our backbone points using `map_traffic_to_points`.
**Datasets Used**: `gdf_points`, `gdf_traffic`.
**Main Assumptions**:
- *Assumption*:
    - `buffer_radius_m` = 50. all traffic route geometries that passes by a 50m radius from the backbone road point is considered as traffic that passes through that backbone road point. This is important since the gdf_traffic has different geometries like the two sides of a highway.
    - We do not take into account the traffic that passes by the point but has a different direction, like a bridge intersection. To filter out this traffic we ensure that each traffic road geometry has to touch at least 2 consecutive points of the backbone road points.
- *Rationale*: Captures traffic from the correct road even with slight GPS inaccuracies.


In [21]:
print(f"buffer_radius_m: {params['buffer_radius_m']}, traffic_columns:{params['traffic_columns']}")

buffer_radius_m: 50, traffic_columns:['total_max', 'short_max', 'medio_max']


In [22]:
gdf_points = map_traffic_to_points(
    gdf_points,
    gdf_traffic,
    params['traffic_columns'],
    params['buffer_radius_m']
)
gdf_points[params['traffic_columns']].describe()


 - Mapping traffic columns ['total_max', 'short_max', 'medio_max'] (Buffer=50m)...
   Mapping columns: ['total_max', 'short_max', 'medio_max']


,total_max,short_max,medio_max
count,147288.000000,147288.000000,147288.000000
mean,32861.475829,7600.897533,12633.115880
std,47627.030784,19795.059084,20326.373441
min,0.000000,0.000000,0.000000
25%,288.410000,119.810000,39.750000
50%,12477.780000,1769.810000,4757.690000
75%,47696.297500,5890.140000,15662.060000
max,628705.660000,546100.160000,318188.120000


### Map 1: Road & Traffic
Points are colored based on their traffic intensity (`total_max`). Green indicates light traffic, yellow mid traffic, and red heavy traffic.

**Visualization**: Backbone points colored by traffic intensity.


In [ ]:
import branca.colormap as cm

m1 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=gdf_points['total_max'].quantile(0.95))
colormap.caption = 'Traffic Intensity (Total Max)'
colormap.add_to(m1)

gdf_plot = gdf_points.to_crs(4326)
for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=colormap(row['total_max']),
        fill=True,
        tooltip=f"Traffic: {row['total_max']}"
    ).add_to(m1)
m1

Output hidden; open in https://colab.research.google.com to view.

### 2.3 Existing Infrastructure Proximity
**Procedure**: Calculating the distance to the nearest existing EV charging station for every point.
**Datasets Used**: `gdf_points`, `gdf_chargers`.
**Main Assumptions**:
- *Assumption*: `max_distance` = 100km.  
Any point of the backbone road points is mapped to its nearest EV Charging station, but if the distance to the nearest charging station is above 100km that point is selected as a point that doesn't have a 'near' charging station. This max parameter is created to prevent outliers generation.


In [23]:
params['max_distance_proximity']

100000

In [24]:
gdf_points = assign_nearest_charging_stations(
    gdf_points = gdf_points,
    gdf_chargers = gdf_chargers,
    max_distance = params['max_distance_proximity']
)

 - Assigning nearest charging stations (MaxDist=100000)...


**Procedure**: Calculating the distance to the nearest traditional gas station (potential conversion sites).
**Datasets Used**: `gdf_points`, `gdf_gas`.
**Main Assumptions**:
- *Assumption*: `max_distance` = 100km.  
Any point of the backbone road points is mapped to its nearest traditional gas station, but if the distance to the nearest traditional gas station is above 100km that point is selected as a point that doesn't have a 'near' traditional gas station. This max parameter is created to prevent outliers generation.


In [25]:
gdf_points = assign_nearest_gas_stations(
    gdf_points = gdf_points,
    gdf_gas = gdf_gas,
    max_distance = params['max_distance_proximity']
)
print("Proximity analysis completed.")

 - Assigning nearest gas stations (MaxDist=100000)...
Proximity analysis completed.


### Map 2: Road & Current EV Chargers
This map highlights the areas with existing infrastructure. Points are colored by distance to the nearest charger (Green = Close, Red = Far).

**Visualization**: Points colored by their distance to the nearest existing charger (identifying gaps).


In [ ]:
gdf_plot = gdf_points.to_crs(4326)

import branca.colormap as cm

m2 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
dist_colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=max(gdf_plot['dist_charger_m']))
dist_colormap.caption = 'Distance to Nearest Charger (m)'
dist_colormap.add_to(m2)

# Explicitly fill NaN values in 'dist_charger_m' before iteration
gdf_plot['dist_charger_m'] = gdf_plot['dist_charger_m'].fillna(max(gdf_plot['dist_charger_m']))

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=dist_colormap(min(row['dist_charger_m'], 100000)), # Use the cleaned column directly
        fill=True
    ).add_to(m2)
m2

Output hidden; open in https://colab.research.google.com to view.

# Phase 3: Grid-Aware Optimization <a name="phase-3"></a>

Solving for the optimal location of new charging sites while considering electrical grid constraints.


### 3.1 Optimization Setup
**Procedure**: Importing the grid-aware optimization engine.


In [26]:
from scripts.processing.optimize_grid_aware_placement import generate_smart_candidates, solve_grid_aware_optimization, report

**Procedure**: Extracting the 2027 EV penetration percentage for the model.
- *Assumption*: For our optimization model, we assume that out of the total traffic on each road point, only a '2027 EV penetration percentage' would be from Electric Vehicles

In [27]:
ev_traffic_pct = vehicle_registrations_2027['pct_ev_registrations'].iloc[0]
print(ev_traffic_pct)

0.13902979980458371


**Procedure**: Calculating the ratio of medium-distance traffic that likely needs charging.
- *Assumption*: For our optimization model, we assume that from the total traffic of any point, only the traffic that passes by that point for a medium route segment is the one that needs to recharge at this point. The short route segment traffic don't need to recharge, while the long route segment traffic is not from electric vehicles.

- *Final Assumption*: The amount of cars that needs to recharge at any specific point i would be
    - EV_needs_recharge_i = Total_traffic_i  * ev_traffic_pct *  need_charge_pct

In [28]:
need_charge_pct = gdf_points['medio_max'].sum() / gdf_points['total_max'].sum()
need_charge_pct

np.float64(0.384435438796252)

**Procedure**: Setting the limit for chargers per site based on existing maximums.
- *Assumption*: For our optimization model, we set that each charging station would have a maximum chargers, limited by the existing max infrastructure of a charging station. This prevents having a charging station with more charging points that the infrastructure could potentially handle (for example a charging station with 200 chargers)   


In [29]:
max_chargers_per_site =  gdf_chargers['charger_count'].max()
max_chargers_per_site

np.int64(30)

### 3.2 Optimization Parameters & Assumptions
**Procedure**: Defining the constraints and penalties for the MILP model.
**Main Assumptions**:
- `coverage_threshold_m` (30km): Maximum distance a user should travel to a charger. The EU regulations state that each charging station should be max 60km away from the next charging station, max distance between charging stations should be 60km. To meet this requirement at any specific point of the road the next charging station should be max at 30km (60km/2).  
- `sessions_per_charger` (24): Maximum charging sessions a single charger can handle per day. The average ultra fast sessions from 20%-80% of the batery of a EV could last between 20-40 minutes, plus a possible no usage rate, we calculate that each session would last 1h, so in a day a charger would have 24 sessions.
- `substation_threshold_m` (10km): Maximum distance to connect a site to a substation. Due to physical restrictions and feasibility, a charging station should be supplied by the nearest electric substation with a max distance of 10km between them. if the distance between the station and the substation is greater than 10km the connection is not possible.
- `power_per_charger_kw` (150kW): Assumed power of new ultra-fast chargers. power demand for each ultra fast charger

- `Optimization penalties (restrictions)`
    - `penalty_coverage`: Cost of leaving a demand point uncovered.
    - `penalty_supply`: Cost of unmet charging demand.
    - `penalty_grid_upgrade`: Priority for sites that don't require immediate grid upgrades.
    - `penalty_coverage` > `penalty_supply` > `penalty_grid_upgrade`

In [30]:
coverage_threshold_m = 30000
sessions_per_charger = 24
substation_threshold_m = 10000
power_per_charger_kw = 150
penalty_coverage = 1e6
penalty_supply = 1e4
penalty_grid_upgrade = 1e2

**Procedure**: Extracting the 2027 EV penetration percentage for the model.


In [31]:
df_cand = generate_smart_candidates(
    gdf_backbone = gdf_points,
    gdf_chargers = gdf_chargers,
    gdf_gas = gdf_gas,
    ev_traffic_pct = ev_traffic_pct,
    need_charge_pct = need_charge_pct,
    coverage_threshold_m = coverage_threshold_m,
    max_chargers_per_site = max_chargers_per_site,
    sessions_per_charger = sessions_per_charger
    )

📍 Generating Smart Candidates...


**Procedure**: run the optimization process.


In [32]:
df_res, grid_slacks = solve_grid_aware_optimization(
        gdf_backbone = gdf_points,
        df_cand = df_cand,
        gdf_grid = gdf_capacity,
        ev_traffic_pct = ev_traffic_pct,
        need_charge_pct= need_charge_pct,
        coverage_threshold_m = coverage_threshold_m,
        substation_threshold_m = substation_threshold_m,
        max_chargers_per_site = max_chargers_per_site,
        sessions_per_charger = sessions_per_charger,
        power_per_charger_kw = power_per_charger_kw,
        penalty_coverage = penalty_coverage,
        penalty_supply = penalty_supply,
        penalty_grid_upgrade = penalty_grid_upgrade
    )



🧠 Formulating Grid-Aware optimization (Vars: 323823)...
   Building road constraints...
   Building grid constraints...
🚀 Solving Grid-Aware Linear Relaxation...


/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/scripts/processing/optimize_grid_aware_placement.py:172: RuntimeWarning: Unrecognized options detected: {'random_seed'}. These will be passed to HiGHS verbatim.
  res = milp(


   Optimization Successful! (Time: 12.95s)


### 3.3 Results Summary
**Procedure**: Generating a high-level report of the recommended infrastructure roadmap.


In [33]:
report(df_res, grid_slacks)


📊 --- GRID-AWARE OPTIMIZATION SUMMARY ---
Total Stations Recommendation = 2378
Total New Chargers = 42151
Substations requiring UPGRADES = 767
Total Grid Capacity Gap = 5020940 kW
  - Grid Upgrade Required: 1865
  - Feasible: 513
-------------------------------------------



In [34]:
gdf_res = gpd.GeoDataFrame(df_res, geometry='geometry', crs=gdf_points.crs)
gdf_res.to_parquet("data/processed/grid_aware_optimized_sites.parquet")

In [35]:
gdf_res

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility
0,0ABYOR885E4US6RFRJFV,existing,4,POINT (451056.608 4559802.443),2193,NaN,inf,0,4,0,No Grid Access (>10km)
1,0MSSBLUV4UTE7AE2J1B6,existing,2,POINT (584948.394 4682782.607),1607,R1-001_4629,1370.993323,1,30,28,Grid Upgrade Required
2,0OW3YBKDKJUTPB7ALJGO,existing,2,POINT (811005.244 4735727.009),1795,R1-299_4587714,220.657489,1,30,28,Grid Upgrade Required
3,0S9SNBBYAX5TI0YO5JKW,existing,2,POINT (984035.631 4710815.85),2193,NaN,inf,0,2,0,No Grid Access (>10km)
4,0U2UXOGZDVMXLO61BBCZ,existing,2,POINT (523465.844 4747233.793),148,R1-001_3017,2403.488143,1,4,2,Grid Upgrade Required
...,...,...,...,...,...,...,...,...,...,...,...
13522,GF_ID_190987_484,greenfield,0,POINT Z (185510.523 4661123.4 1239.044),2193,NaN,inf,0,0,0,No Grid Access (>10km)
13523,GF_ID_190987_488,greenfield,0,POINT Z (184722.253 4661221.652 1278.157),2193,NaN,inf,0,0,0,No Grid Access (>10km)
13524,GF_ID_190987_490,greenfield,0,POINT Z (184338.284 4661117.535 1308.832),2193,NaN,inf,0,0,0,No Grid Access (>10km)
13525,GF_ID_190987_581,greenfield,0,POINT Z (167232.975 4661846.065 1046.381),2193,NaN,inf,0,0,0,No Grid Access (>10km)


# Phase 4: Output Construction <a name="phase-4"></a>

In this final phase, we process the optimization results to generate a clean, actionable dataset. This involves calculating infrastructure requirements, enriching the data with spatial attributes (coordinates and road names), and filtering the results to focus on the newly proposed charging locations.


### 4.1 Capacity Estimation
**Procedure**: Calculating the total estimated power demand (kW) for each site based on the number of chargers.
**Datasets Used**: `gdf_res`.
**Main Assumptions**:
- *Assumption*: Power per charger = 150 kW.
- *Rationale*: We target ultra-fast charging to minimize user wait times and maximize turnover at each site.


In [36]:
gdf_res['estimated_demand_kw'] = gdf_res['final_n'] * 150

### 4.2 Coordinate Extraction
**Procedure**: Extracting Latitude and Longitude from the point geometries.
**Datasets Used**: `gdf_res`.
**Rationale**: Explicit coordinates are required for integration with GIS tools, mapping APIs, and downstream reporting formats.


In [37]:
gdf_res['latitude'] = gdf_res.geometry.y
gdf_res['longitude'] = gdf_res.geometry.x

### 4.3 Result Filtering & Investment Isolation
**Procedure**: Filtering the optimization results to isolate sites where infrastructure was successfully placed.
**Datasets Used**: `gdf_res` (the optimization output that has all possible candidates to create a charging station, wether its selected as a EV charging station or not)
**Rationale (The "Why")**:
1. **Operational Focus**: We filter for `is_open == 1` to ignore candidates that the model rejected due to low demand or grid constraints.
2. **New Infrastructure identification**: By further filtering for `type != 'existing'`, we create a specific list of **newly proposed sites** (at gas stations or greenfields). This is critical for budgeting and project management, as it separates maintenance of current sites from new construction initiatives.


In [38]:
df_stations = gdf_res[(gdf_res['is_open']==1)&(gdf_res['type']!='existing')]

In [39]:
gdf_res[(gdf_res['is_open']==1)]

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude
1,0MSSBLUV4UTE7AE2J1B6,existing,2,POINT (584948.394 4682782.607),1607,R1-001_4629,1370.993323,1,30,28,Grid Upgrade Required,4500,4.682783e+06,584948.394006
2,0OW3YBKDKJUTPB7ALJGO,existing,2,POINT (811005.244 4735727.009),1795,R1-299_4587714,220.657489,1,30,28,Grid Upgrade Required,4500,4.735727e+06,811005.243836
4,0U2UXOGZDVMXLO61BBCZ,existing,2,POINT (523465.844 4747233.793),148,R1-001_3017,2403.488143,1,4,2,Grid Upgrade Required,600,4.747234e+06,523465.844158
5,14OHI9EMAHFI3X4WQ1YU,existing,8,POINT (903508.229 4618428.296),1261,R1-299_4644671,793.253345,1,8,0,Feasible,1200,4.618428e+06,903508.228737
6,1BEIJQUXN6N3M30WQUCC,existing,4,POINT (236833.558 4141663.271),1448,R1-299_510006035,915.309228,1,4,0,Feasible,600,4.141663e+06,236833.557735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13400,1529,gas,0,POINT (646591.256 4614169.84),93,R1-299_11007070,2561.083549,1,30,30,Grid Upgrade Required,4500,4.614170e+06,646591.256391
13409,9079,gas,0,POINT (714485.711 4583650.834),1484,R1-299_1250056,5181.799604,1,30,30,Grid Upgrade Required,4500,4.583651e+06,714485.710703
13500,1452,gas,0,POINT (683109.178 4636252.075),936,R1-299_15164019,1779.753072,1,13,13,Grid Upgrade Required,1950,4.636252e+06,683109.177899
13502,GF_ID_190375_106,greenfield,0,POINT Z (853243.04 4699374.923 1298.935),132,R1-299_4244466,9971.315765,1,30,30,Grid Upgrade Required,4500,4.699375e+06,853243.039826


### 4.4 Unique Site Identification
**Procedure**: Assigning a unique, human-readable ID (`IBE_XXX`) to each proposed location.
**Rationale**: Standardized IDs simplify tracking, permit identification, and stakeholder reporting.


In [40]:
df_stations['location_id'] = [f"IBE_{i:03d}" for i in range(1, len(df_stations) + 1)]

### 4.5 Spatial Merge & Road Contextualization
**Procedure**: Performing a **nearest spatial join** (effectively a spatial merge) between our proposed sites and the original road backbone.
**Datasets Used**: `df_stations`, `gdf_points`.
**Rationale (The "Why")**:
1. **Actionable Mapping**: While the model optimizes points in space, stakeholders (engineers, permit managers) need to know which highway each site serves. This "merge" maps coordinates back to human-readable road names (e.g., A-2, AP-7).
2. **Data Enrichment**: This step ensures that the final output contains all necessary metadata from the initial backbone analysis, bridging the gap between raw spatial optimization and infrastructure planning.


In [41]:
df_stations_indexed = df_stations.set_index('location_id')

df_stations_with_road = gpd.sjoin_nearest(
    df_stations_indexed,
    gdf_points[['road_name', 'geometry']],
    how='left',
)

df_stations_with_road = df_stations_with_road.loc[~df_stations_with_road.index.duplicated(keep='first')]

df_stations_with_road.rename(columns={'road_name': 'nearest_road_name'}, inplace=True)

df_stations = df_stations.set_index('location_id')
df_stations['route_segment'] = df_stations_with_road['nearest_road_name']
df_stations = df_stations.reset_index()


In [42]:
df_stations

,location_id,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude,route_segment
0,IBE_001,4383,gas,0,POINT (658248.244 4304489.892),1287,R1-001_4262,4967.091944,1,4,4,Grid Upgrade Required,600,4.304490e+06,658248.243575,A-31
1,IBE_002,9035,gas,0,POINT (652793.516 4313691.606),440,R1-001_4243,2510.186304,1,4,4,Grid Upgrade Required,600,4.313692e+06,652793.515664,A-31
2,IBE_003,9254,gas,0,POINT (569829.947 4322772.158),151,R1-001_3590,3370.664962,1,4,4,Grid Upgrade Required,600,4.322772e+06,569829.947278,N-430
3,IBE_004,5282,gas,0,POINT (540286.827 4311703.79),1530,R1-001_3591,784.996461,1,19,19,Grid Upgrade Required,2850,4.311704e+06,540286.827206,N-430
4,IBE_005,5142,gas,0,POINT (678808.933 4288543.12),710,R1-001_3139,1907.854468,1,4,4,Feasible,600,4.288543e+06,678808.933326,N-344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1256,IBE_1257,1529,gas,0,POINT (646591.256 4614169.84),93,R1-299_11007070,2561.083549,1,30,30,Grid Upgrade Required,4500,4.614170e+06,646591.256391,AP-68
1257,IBE_1258,9079,gas,0,POINT (714485.711 4583650.834),1484,R1-299_1250056,5181.799604,1,30,30,Grid Upgrade Required,4500,4.583651e+06,714485.710703,N-232
1258,IBE_1259,1452,gas,0,POINT (683109.178 4636252.075),936,R1-299_15164019,1779.753072,1,13,13,Grid Upgrade Required,1950,4.636252e+06,683109.177899,N-330
1259,IBE_1260,GF_ID_190375_106,greenfield,0,POINT Z (853243.04 4699374.923 1298.935),132,R1-299_4244466,9971.315765,1,30,30,Grid Upgrade Required,4500,4.699375e+06,853243.039826,N-260


### 4.6 Grid Capacity Enrichment
**Procedure**: Merging the proposed charging sites with the electric grid capacity dataset to retrieve technical substation metadata.
**Datasets Used**: `df_stations`, `gdf_capacity`.
**Rationale**: To evaluate the feasibility of our infrastructure roadmap, we must associate every proposed site with the actual electrical limits of its connecting substation. This **merge** links each site's `substation_id` to its available `capacity_kw` and operating `company`.


In [43]:
capacity_usefull_cols = gdf_capacity[['company','row_id','capacity_kw']]

df_stations = df_stations.merge(
    capacity_usefull_cols,
    left_on = 'substation_id',
    right_on = 'row_id',
    how = 'left'
    )

df_stations.head()

,location_id,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude,route_segment,company,row_id,capacity_kw
0,IBE_001,4383,gas,0,POINT (658248.244 4304489.892),1287,R1-001_4262,4967.091944,1,4,4,Grid Upgrade Required,600,4.304490e+06,658248.243575,A-31,Iberdrola,R1-001_4262,0.0
1,IBE_002,9035,gas,0,POINT (652793.516 4313691.606),440,R1-001_4243,2510.186304,1,4,4,Grid Upgrade Required,600,4.313692e+06,652793.515664,A-31,Iberdrola,R1-001_4243,0.0
2,IBE_003,9254,gas,0,POINT (569829.947 4322772.158),151,R1-001_3590,3370.664962,1,4,4,Grid Upgrade Required,600,4.322772e+06,569829.947278,N-430,Iberdrola,R1-001_3590,0.0
3,IBE_004,5282,gas,0,POINT (540286.827 4311703.79),1530,R1-001_3591,784.996461,1,19,19,Grid Upgrade Required,2850,4.311704e+06,540286.827206,N-430,Iberdrola,R1-001_3591,0.0
4,IBE_005,5142,gas,0,POINT (678808.933 4288543.12),710,R1-001_3139,1907.854468,1,4,4,Feasible,600,4.288543e+06,678808.933326,N-344,Iberdrola,R1-001_3139,46040.0


### 4.7 Grid Stress Analysis & Feasibility Status
**Procedure**: Aggregating the total proposed demand at the substation level and calculating a **Capacity-to-Demand Ratio** to determine grid status.
**Datasets Used**: `df_stations`.
**Main Assumptions (Logic)**:
- **Ratio Calculation**: `capacity_kw / estimated_demand_kw`.
- **Status Classification**:
    - **Congested**: Ratio > 1.2 or 0 (High demand relative to capacity, 20% more than the available capacity, or 0 capacity).
    - **Moderate**: Ratio > 1 (Demand is higher than the available capacity but lower than a 20% more than the available capacity).
    - **Sufficient**: Ratio <= 1 (Capacity exceeds demand).
**Rationale**: This analysis identifies potential grid bottlenecks. By calculating the ratio of available capacity to the new infrastructure load, we can prioritize sites that are "grid-ready" and flag those requiring significant reinforcement.


In [44]:
subs_demand = df_stations.groupby('substation_id')[['estimated_demand_kw','capacity_kw']].agg({'estimated_demand_kw':'sum','capacity_kw':'max'}).reset_index()

subs_demand['ratio'] = subs_demand['capacity_kw']/subs_demand['estimated_demand_kw']
subs_demand['grid_status'] = np.where(
    (subs_demand['ratio'] >1.2)|(subs_demand['ratio']==0), 'Congested',
    np.where(subs_demand['ratio']>1, 'Moderate', 'Sufficient')
)

df_final = df_stations.merge(
    subs_demand[['substation_id','ratio','grid_status']],
    on = 'substation_id',
    how = 'left'
    )

In [45]:
file_2 = df_final[['location_id','latitude','longitude','route_segment','final_n','grid_status']]
file_2.rename(columns = {'final_n': 'n_chargers_proposed'},inplace = True)
file_2.head()

,location_id,latitude,longitude,route_segment,n_chargers_proposed,grid_status
0,IBE_001,4.304490e+06,658248.243575,A-31,4,Congested
1,IBE_002,4.313692e+06,652793.515664,A-31,4,Congested
2,IBE_003,4.322772e+06,569829.947278,N-430,4,Congested
3,IBE_004,4.311704e+06,540286.827206,N-430,19,Congested
4,IBE_005,4.288543e+06,678808.933326,N-344,4,Congested


In [46]:
file_2.to_csv('data/outputs/File 2.csv')

In [47]:
file_3 = df_final[['latitude','longitude','route_segment','company','estimated_demand_kw','grid_status']]
file_3 = file_3[file_3['grid_status']!= 'Sufficient']
file_3['bottleneck_id'] = [f"FRIC_{i:03d}" for i in range(1, len(file_3) + 1)]
file_3 = file_3[['bottleneck_id','latitude','longitude','route_segment','company','estimated_demand_kw','grid_status']]
file_3.rename(columns = {'company': 'distributor_network'}, inplace = True )
file_3.head()

,bottleneck_id,latitude,longitude,route_segment,distributor_network,estimated_demand_kw,grid_status
0,FRIC_001,4.304490e+06,658248.243575,A-31,Iberdrola,600,Congested
1,FRIC_002,4.313692e+06,652793.515664,A-31,Iberdrola,600,Congested
2,FRIC_003,4.322772e+06,569829.947278,N-430,Iberdrola,600,Congested
3,FRIC_004,4.311704e+06,540286.827206,N-430,Iberdrola,2850,Congested
4,FRIC_005,4.288543e+06,678808.933326,N-344,Iberdrola,600,Congested


In [48]:
file_3.to_csv('data/outputs/File 3.csv')

In [49]:
file_1 = pd.DataFrame({
    'total_proposed_stations': [len(file_2)],
    'total_exisiting_stations_baseline': [len(gdf_chargers)],
    'total_friction_points': [len(file_3)],
    'total_ev_projected_2027' :  [vehicle_registrations_total['total_ev_registrations'].item()]
})
file_1

,total_proposed_stations,total_exisiting_stations_baseline,total_friction_points,total_ev_projected_2027
0,1261,1298,1221,664995.0


In [50]:
file_1.to_csv('data/outputs/File 1.csv')

In [51]:

# 1. Map Colors
color_map = {
    'Sufficient': 'green',
    'Moderate': 'yellow',
    'Congested': 'red'
}

# 2. Coordinate Transformation (Optional: only if coordinates are still in projected meters)
# Assuming source is EPSG:3042 (Spain UTM) and converting to WGS84 for Folium
if 'geometry' in file_2.columns and isinstance(file_2, gpd.GeoDataFrame):
    df_map = file_2.to_crs(epsg=4326)
else:
    # If it's a CSV with X/Y columns in meters, create GeoDataFrame first
    gdf = gpd.GeoDataFrame(
        file_2,
        geometry=gpd.points_from_xy(file_2.longitude, file_2.latitude),
        crs="EPSG:3042"
    )
    df_map = gdf.to_crs(epsg=4326)

# 3. Initialize Map
m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')

# 4. Add Points to Map
for _, row in df_map.iterrows():
    # Use Tooltip for hover information
    tooltip_html = f"""
        <b>Route Segment:</b> {row['route_segment']}<br>
        <b>Proposed Chargers:</b> {row['n_chargers_proposed']}<br>
        <b>Grid Status:</b> {row['grid_status']}<br>
        <b>Coordinates:</b> ({row.geometry.y:.5f}, {row.geometry.x:.5f})
    """

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color_map.get(row['grid_status'], 'gray'),
        fill=True,
        fill_color=color_map.get(row['grid_status'], 'gray'),
        fill_opacity=0.7,
        tooltip=tooltip_html  # Shows on hover
    ).add_to(m)

# Display Map
m.save("grid_status_map.html") # Or just 'm' in a notebook

m


